## Kernel to load: vax_inc_general 

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import binomtest
import pycountry
from functools import reduce
import random
import pycountry
import sys,os
import warnings
import re
from collections import defaultdict
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [2]:
notebook_dir = os.path.dirname(os.getcwd())
source_data_path=os.path.join(notebook_dir, "Common Source Data")
sys.path.append(source_data_path)
from country_codes import countries

In [3]:
df_start= pd.read_csv(os.path.join(source_data_path, "WAHIS data","cattle_country-wide_vaccinated.csv"))
df_start=df_start[df_start['Semester']!='Jul-Dec 2025']
df_start

,Year,Semester,Region,Country,Disease,Animal category,Species,Vaccine type,Number of vaccinated
0,2005,Jul-Dec 2005,Asia,Afghanistan,Haemorrhagic septicaemia (Pasteurella multocid...,Domestic,Cattle,-,14100
1,2005,Jul-Dec 2005,Asia,Afghanistan,Foot and mouth disease virus (Inf. with),Domestic,Cattle,-,46069
2,2005,Jul-Dec 2005,Asia,Afghanistan,Anthrax,Domestic,Cattle,-,50182
3,2005,Jul-Dec 2005,Europe,Albania,Anthrax,Domestic,Cattle,-,-
4,2005,Jul-Dec 2005,Africa,Algeria,Rabies virus (Inf. with),Domestic,Cattle,-,26593
...,...,...,...,...,...,...,...,...,...
14843,2025,Jan-Jun 2025,Europe,Portugal,Bluetongue virus (Inf. with),Domestic,Cattle,-,-
14844,2025,Jan-Jun 2025,Europe,Portugal,Brucella abortus (Inf. with),Domestic,Cattle,-,-
14845,2025,Jan-Jun 2025,Americas,Uruguay,Brucella abortus (Inf. with),Domestic,Cattle,Live Attenuated Vaccine,92006
14846,2025,Jan-Jun 2025,Americas,Uruguay,Brucella abortus (Inf. with),Domestic,Cattle,-,-


In [4]:
# List of diseases where no vaccine exists
diseases_no_vaccine = [
    "African cattle fever virus (Inf. with)",
    "Avian tuberculosis (-2005)",
    "Bovine spongiform encephalopathy",
    "Crimean Congo haemorrhagic fever (2006-)",
    "Maedi-visna",
    "Malignant catarrhal fever (wildebeest only)(2006-2008)",
    "New world screwworm (Cochliomyia hominivorax)",
    "Nipah virus encephalitis",
    "Scrapie",
    "Surra (Trypanosoma evansi)",
    "Theileria equi and Babesia caballi (Inf. with) (Equine piroplasmosis)",
    "Tularemia"
]


In [5]:
records_edited = 0
diseases_found = []
removed_records = []  

# Iterate through diseases and remove invalid records
for disease in diseases_no_vaccine:
    if disease in df_start['Disease'].unique():
        diseases_found.append(disease)  
        condition = (df_start['Disease'] == disease)
        
        disease_removed_records = df_start[condition]
        removed_records.append(disease_removed_records)
        
        count_disease_edits = disease_removed_records.shape[0]
        records_edited += count_disease_edits
        
        df_start = df_start[~condition]

print(f"Total records edited (removed): {records_edited}")

if diseases_found:
    print("\nDiseases with removed vaccination records:")
    for d in diseases_found:
        print(f" - {d}")
else:
    print("\nNone of the specified diseases were found in the dataframe.")

if removed_records:
    print("\nDetails of removed records:")
    for i, records in enumerate(removed_records):
        print(f"\nRemoved records for disease: {diseases_found[i]}")
        print(records)
else:
    print("\nNo records were removed.")

df_start.reset_index(drop=True, inplace=True)
print("\nFinal dataframe shape:", df_start.shape)


Total records edited (removed): 0

None of the specified diseases were found in the dataframe.

No records were removed.

Final dataframe shape: (14848, 9)


In [6]:
df_part_full=pd.read_csv('2005-2025_part_cattle_vaccine_coverage_by_country.csv')

In [7]:
dict_dates = dict({
                  'Jan-Jun 2005':'2005-06-30','Jul-Dec 2005':'2005-12-31',
                  'Jan-Jun 2006':'2006-06-30','Jul-Dec 2006':'2006-12-31',
                  'Jan-Jun 2007':'2007-06-30','Jul-Dec 2007':'2007-12-31',
                  'Jan-Jun 2008':'2008-06-30','Jul-Dec 2008':'2008-12-31',
                  'Jan-Jun 2009':'2009-06-30','Jul-Dec 2009':'2009-12-31',
                  'Jan-Jun 2010':'2010-06-30','Jul-Dec 2010':'2010-12-31',
                  'Jan-Jun 2011':'2011-06-30','Jul-Dec 2011':'2011-12-31',
                  'Jan-Jun 2012':'2012-06-30','Jul-Dec 2012':'2012-12-31',
                  'Jan-Jun 2013':'2013-06-30','Jul-Dec 2013':'2013-12-31',
                  'Jan-Jun 2014':'2014-06-30','Jul-Dec 2014':'2014-12-31',
                  'Jan-Jun 2015':'2015-06-30','Jul-Dec 2015':'2015-12-31',
                  'Jan-Jun 2016':'2016-06-30','Jul-Dec 2016':'2016-12-31',
                  'Jan-Jun 2017':'2017-06-30','Jul-Dec 2017':'2017-12-31',
                  'Jan-Jun 2018':'2018-06-30','Jul-Dec 2018':'2018-12-31',
                  'Jan-Jun 2019':'2019-06-30','Jul-Dec 2019':'2019-12-31',
                  'Jan-Jun 2020':'2020-06-30','Jul-Dec 2020':'2020-12-31',
                  'Jan-Jun 2021':'2021-06-30','Jul-Dec 2021':'2021-12-31',
                  'Jan-Jun 2022':'2022-06-30','Jul-Dec 2022':'2022-12-31',
                  'Jan-Jun 2023':'2023-06-30','Jul-Dec 2023':'2023-12-31',
                  'Jan-Jun 2024':'2024-06-30','Jul-Dec 2024':'2024-12-31',
                  'Jan-Jun 2025':'2025-06-30'
                                                                        })

df_start=df_start.copy()

df_start=df_start.copy()
df_start['time']=[dict_dates[i] for i in df_start['Semester']]
df_start['Semester']=['June' if '06-30' in time else 'December' for time in df_start['time'].values]
df_start['time'] = pd.to_datetime(df_start['time'])

In [8]:
df_start=df_start.copy()

df_start['Number of vaccinated']=[float(i) if i!='-' else None for i in df_start['Number of vaccinated']]

df_start=df_start[df_start['Number of vaccinated']>=0]
df_start['Year Range']=df_start['Year']

In [9]:
#Adjustments for miscategorized reports
df_start.loc[(df_start['Disease']=='High pathogenicity avian influenza viruses (poultry) (Inf. with)')&
(df_start['Species']=='Cattle')&(df_start['Year']>=2017),'Disease']='Influenza A viruses of high pathogenicity (Inf. with) (non-poultry including wild birds) (2017-)'

df_start.loc[(df_start['Disease']=='High pathogenicity avian influenza viruses (poultry) (Inf. with)')&
(df_start['Species']=='Cattle')&(df_start['Year']<2017),'Disease']='Influenza A virus (Inf. with)'

df_start.loc[(df_start['Disease']=='Low pathogenic avian influenza (poultry) (2006-2021)')&
(df_start['Species']=='Cattle'),'Disease']='Influenza A virus (Inf. with)'

In [10]:
start_countries=df_start['Country']

codes_start=[countries.get(country, 'Unknown code:'+country) for country in start_countries]

for code in codes_start:
    if "Unknown" in code:
        print("FIX THIS:",code)


iso3s_start=[]

for i in start_countries:
    try:
        iso3s_start+=[countries[i]]
    except:
        iso3s_start+=[None]

df_start=df_start.copy()
     
df_start['ISO3']=iso3s_start

In [11]:
 # Aggregate rows by 'Country', 'Disease', and 'Year' to sum 'Number of vaccinated'
df_start = df_start.groupby(['Year', 'Country', 'Disease'], as_index=False).agg({
    'ISO3': 'first',
    'Country': 'first',
    'Year': 'first',
    'Region': 'first',
    'Disease': 'first',
    'Animal category': 'first',
    'Species': 'first',
    'Vaccine type': 'first',
    'Number of vaccinated': 'sum', 
    'time': 'first', 
})


pop_cattle_df = pd.read_csv(os.path.join(source_data_path, 'Processed data','FAO Populations','cattle_pop.csv')).loc[:,['Area','Unit','Value','Year','Item','ISO3']]
pop_cattle_df = pop_cattle_df.sort_values('Value').drop_duplicates(subset=['ISO3','Year','Item'], keep='last')
pop_cattle_df.rename(columns={'Value':'TOTAL Population'},inplace=True)

pop_cattle_df = (
    pop_cattle_df.groupby(['Area', 'Year','ISO3'], as_index=False)
    .agg({
        'ISO3':'first',
        'Area': 'first',
        'Year': 'first',
        'Unit': 'first',
        'Item': 'first',
        'TOTAL Population': 'sum',  
    })
)
pop_cattle_df.drop(columns=['Item'],inplace=True)


killed_pop_cattle_df = pd.read_csv(os.path.join(source_data_path, 'Processed data','FAO Populations','killed_cattle_pop.csv')).loc[:,['Area','Unit','Value','Year','Item','ISO3']]
killed_pop_cattle_df = killed_pop_cattle_df.sort_values('Value').drop_duplicates(subset=['ISO3','Year','Item'], keep='last')
killed_pop_cattle_df.rename(columns={'Value':'TOTAL Slaughtered Population'},inplace=True)
killed_pop_cattle_df = (
    killed_pop_cattle_df.groupby(['Area', 'Year','ISO3'], as_index=False)
    .agg({
        'ISO3':'first',
        'Area': 'first',
        'Year': 'first',
        'Unit': 'first',
        'Item': 'first',
        'TOTAL Slaughtered Population': 'sum',  
    })
)


pop_cattle_df = pop_cattle_df.sort_values('TOTAL Population').drop_duplicates(subset=['ISO3','Year'], keep='last')
killed_pop_cattle_df = killed_pop_cattle_df.sort_values('TOTAL Slaughtered Population').drop_duplicates(subset=['ISO3','Year'], keep='last')


df_start=reduce(lambda  left,right: pd.merge(left,right,on=['ISO3','Year'],
                                                how='left'), [df_start,
                                                             pop_cattle_df.drop(columns=['Unit','Area']),
                                                              killed_pop_cattle_df.drop(columns=['Unit','Area','Item'])])

df_start['Vaccine Coverage Intermediate']=df_start['Number of vaccinated']/(df_start['TOTAL Population']+df_start['TOTAL Slaughtered Population'])


In [12]:
# Add missing years for each unique combination of Semester, ISO3, Administrative Division, and Disease
def add_missing_years(df):
    group_columns = ["ISO3", "Disease"]

    unique_combinations = df[group_columns].drop_duplicates()
    unique_combinations['Min_Year'] = df.groupby(group_columns)['Year'].transform('min')
    unique_combinations['Max_Year'] = df.groupby(group_columns)['Year'].transform('max')

    all_years = []
    for _, row in unique_combinations.iterrows():
        years = pd.DataFrame({'Year': range(row['Min_Year'], row['Max_Year'] + 1)})
        for col in group_columns:
            years[col] = row[col]
        all_years.append(years)

    all_years_df = pd.concat(all_years, ignore_index=True)

    expanded_df = pd.merge(all_years_df, df, on=group_columns + ['Year'], how='left')

    none_columns = ["Vaccine Coverage Intermediate","Number of vaccinated"]
    for col in none_columns:
        expanded_df[col] = expanded_df[col].where(expanded_df[col].notna(), None)

    ffill_columns = expanded_df.columns.difference(none_columns + ['Year'])

    expanded_df[ffill_columns] = expanded_df.sort_values(by=group_columns + ['Year'])[ffill_columns].ffill()

    expanded_df['Derived_Vaccinated_Method'] = "None"

    return expanded_df




# Interpolate Adjusted_Susceptible and Vaccinated
def interpolate_adjusted_and_cases(df):
    group_columns = ["ISO3", "Disease"]

    def interpolate_group(group):
        group = group.sort_values('Year').reset_index(drop=True)

        interpolated_vaccinated = group['Vaccine Coverage Intermediate'].fillna(value=np.nan).interpolate(method='linear')

        group.loc[interpolated_vaccinated.notna() & group['Vaccine Coverage Intermediate'].isna(), 'Derived_Vaccinated_Method'] = "Adjusted_Vaccinated"

        group['Vaccine Coverage Intermediate'] = interpolated_vaccinated

        group['Number of vaccinated']=group['Vaccine Coverage Intermediate'] *(group['TOTAL Population']+group['TOTAL Slaughtered Population'])

        return group

    return df.groupby(group_columns, group_keys=False).apply(interpolate_group)

# Function to collect years data was used for interpolation
def update_interpolated_upper_year(df):
    group_columns = ["ISO3",  "Disease"]

    def assign_upper_year(group):
        group = group.sort_values('Year').reset_index(drop=True)

        for idx in group[group['Derived_Vaccinated_Method'].notna()].index:
            if group.loc[idx, 'Derived_Vaccinated_Method'] in ["Adjusted_Vaccinated"]:
                # Find the next original (non-interpolated) row by year
                upper_idx = group[(group.index > idx) & (group['Derived_Vaccinated_Method']=='None')].index.min()
                lower_idx = group[(group.index < idx) & (group['Derived_Vaccinated_Method']=='None')].index.max()

                
                if pd.notna(upper_idx):  
                    group.loc[idx, 'interpolated_upper_year'] = int(group.loc[upper_idx, 'Year'])
                else:
                    group.loc[idx, 'interpolated_upper_year'] = None  

                if pd.notna(lower_idx): 
                    group.loc[idx, 'interpolated_lower_year'] = int(group.loc[lower_idx, 'Year'])
                else:
                    group.loc[idx, 'interpolated_lower_year'] = None 
        return group

    return df.groupby(group_columns, group_keys=False).apply(assign_upper_year)


def process_dataframe(df):
    df = add_missing_years(df)  

    df.drop(columns=['TOTAL Population','TOTAL Slaughtered Population'],inplace=True)


    df=reduce(lambda  left,right: pd.merge(left,right,on=['ISO3','Year'],
                                                how='left'), [df,
                                                             pop_cattle_df.drop(columns=['Unit','Area']),
                                                              killed_pop_cattle_df.drop(columns=['Unit','Area','Item'])])

    df = interpolate_adjusted_and_cases(df)  # Interpolate Adjusted_Susceptible and Incidence, update Cases
    df=update_interpolated_upper_year(df)
    return df

df_result = process_dataframe(df_start)


In [13]:
def generate_year_range(row):
    if pd.notna(row['interpolated_lower_year']) and pd.notna(row['interpolated_upper_year']):
        return f"{int(row['interpolated_lower_year'])}-{int(row['interpolated_upper_year'])}"
    else:
        return str(int(row['Year']))

df_result['Year Range'] = df_result.apply(generate_year_range, axis=1)


In [14]:
once_lifetime_vaccines=['Brucella abortus (Inf. with)', 'Rinderpest virus (Inf. with)', 'Theileria annulata, Theileria orientalis and Theileria parva (Inf. with)'] 

In [15]:

milk_animals=pd.read_csv(os.path.join(source_data_path,'Processed data','FAO Populations','dairy_cattle_pop.csv'))
milk_animals=milk_animals.loc[:,['Element','Area','Value','Year','ISO3']]
milk_animals.pivot(index=['Area','Year'],columns=['Element'],values=['Value']).reset_index()
milk_animals.drop(columns=['Element'],inplace=True)
milk_animals.columns=['Area','Milk Animals','Year','ISO3']
milk_animals.rename(columns={'Value':'Milk Animals','Area':'Country'},inplace=True)

In [16]:
tot_cows=pd.read_csv(os.path.join(source_data_path, 'Processed data','FAO Populations','cattle_pop.csv'))
tot_cows=tot_cows.loc[:,['Area','Value','Year','ISO3']]
tot_cows.rename(columns={'Value':'Total cattle','Area':'Country'},inplace=True)

In [17]:
pop_milk = milk_animals['Country']
pop_tot_cow=tot_cows['Country']

codes_milk = [countries.get(country, 'Unknown code:'+country) for country in pop_milk]
codes_tot_cow = [countries.get(country, 'Unknown code:'+country) for country in pop_tot_cow]

iso3s_milk=[]

for i in pop_milk:
    try:
        iso3s_milk+=[countries[i]]
    except:
        iso3s_milk+=[None]
        
        
iso3s_value_cow=[]

for i in pop_tot_cow:
    try:
        iso3s_value_cow+=[countries[i]]
    except:
        iso3s_value_cow+=[None]
        
milk_animals['ISO3']=iso3s_milk
tot_cows['ISO3']=iso3s_value_cow

milk_animals=milk_animals.sort_values('Milk Animals').drop_duplicates(['ISO3','Year'],keep='last')
tot_cows=tot_cows.sort_values('Total cattle').drop_duplicates(['ISO3','Year'],keep='last')
milk_animals=pd.merge(milk_animals.drop(columns=['Country']),tot_cows.drop(columns=['Country']),how='left',on=['ISO3','Year'])


In [18]:
milk_animals['prop_milk_cows'] = (
    milk_animals['Milk Animals'] / milk_animals['Total cattle']
).clip(upper=1)
df_result=pd.merge(df_result,milk_animals.drop(columns=['Milk Animals','Total cattle']),how='left',on=['ISO3','Year'])
df_result['prop_milk_cows'] = df_result['prop_milk_cows'].fillna(df_result['prop_milk_cows'].median())

In [19]:
df_result['cattle_lifespan']=(df_result['prop_milk_cows']*5.5)+((1-df_result['prop_milk_cows'])*1.3)

In [21]:
def vaccination_coverage_lifetime_vaccine(df_result):
    """
    Lifespan-adjusted vaccination coverage. For diseases that require once-per-lifetime vaccination (not annual)

    For each country, the code sums the number of cattle vaccinated over the past L years - where L is that country’s average cattle 
    lifespan - and divides by the number of cattle alive in the current year.

    If vaccinated number for past years is missing, it compensates by assigning increased weight to years that have data, with increasing weight
    provided to the years closest to the missing year (i.e., if 2 years are missing, and there is data for one year before, and one year after 
    the three-year hole, then both of the non-missing years have a weight of 2 assigned to them; this means each existing year closest approximates
    for one of the twomissing years). This algorithm compensates for other combinations of possible missing years flexibly to most accurately derive
    a vaccination coverage estimate.

    For each row (ISO3, Disease, Year, cattle_lifespan=L):
      - Construct target years: Y, Y-1, ..., Y-(floor(L)-1).
      - If a target year has no record, split its weight 50/50 to the nearest
        available year on each side (past and future). If only one side is available,
        assign full weight to that side.mm
      - Sum(weights_by_year[yr] * vacc_value[yr]) as adjusted 'Number of vaccinated'.
      - Coverage = adjusted_vacc / (TOTAL Population + TOTAL Slaughtered Population) of the current row.
      - New 'Year Range' = bracketed min-max of the ranges from all used rows; collapse to [single] if equal.
    """

    COL_ISO3   = "ISO3"
    COL_YEAR   = "Year"
    COL_DIS    = "Disease"
    COL_VACC   = "Number of vaccinated"
    COL_POP    = "TOTAL Population"
    COL_SLAUG  = "TOTAL Slaughtered Population"
    COL_RANGE  = "Year Range"
    COL_LIFE   = "cattle_lifespan"

    required = {COL_ISO3, COL_YEAR, COL_DIS, COL_VACC, COL_POP, COL_SLAUG, COL_RANGE, COL_LIFE}
    missing = required - set(df_result.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    df = df_result.copy()

    year_range_rx = re.compile(r'^\s*(\d{1,4})(?:\s*-\s*(\d{1,4}))?\s*$')
    def parse_year_range(s):
        if pd.isna(s):
            return (None, None)
        s = str(s).strip()
        m = year_range_rx.match(s)
        if m:
            y1 = int(m.group(1))
            y2 = int(m.group(2)) if m.group(2) else y1
            return (y1, y2)
        nums = re.findall(r'\d{1,4}', s)
        if nums:
            vals = list(map(int, nums))
            return (min(vals), max(vals))
        return (None, None)

    def format_year_range(lo, hi):
        if lo is None or hi is None:
            return ""
        return f"{lo}" if lo == hi else f"{lo}-{hi}"

    # Prebuild per-(ISO3, Disease) maps
    group_key = [COL_ISO3, COL_DIS]
    grouped = df.groupby(group_key, dropna=False)

    group_info = {}
    for key, g in grouped:
        g = g.dropna(subset=[COL_YEAR]).copy()
        if g.empty:
            group_info[key] = {"years_asc": [], "vacc_by_year": {}, "range_by_year": {}}
            continue
        g.sort_values(COL_YEAR, inplace=True)  # ascending for neighbor search
        yrs = g[COL_YEAR].astype(int).to_numpy()
        # If duplicate years exist, keep the last occurrence after sort (or choose first; consistent usage is key)
        vacc_by_year = {}
        range_by_year = {}
        for _, r in g.iterrows():
            y = int(r[COL_YEAR])
            vacc_by_year[y]  = float(0.0 if pd.isna(r[COL_VACC]) else r[COL_VACC])
            range_by_year[y] = str(r[COL_RANGE])
        group_info[key] = {
            "years_asc": sorted(set(yrs)),
            "vacc_by_year": vacc_by_year,
            "range_by_year": range_by_year,
        }

    def nearest_past(years_asc, t):
        # largest available < t
        idx = np.searchsorted(years_asc, t) - 1
        return years_asc[idx] if idx >= 0 else None

    def nearest_future(years_asc, t, cap=None):
        # smallest available > t (optionally cap at Y)
        idx = np.searchsorted(years_asc, t, side="right")
        while idx < len(years_asc):
            cand = years_asc[idx]
            if cap is None or cand <= cap:
                return cand
            idx += 1
        return None

    def adjust_for_row(row):
        iso = row[COL_ISO3]
        dis = row[COL_DIS]
        Y   = row[COL_YEAR]
        L   = row[COL_LIFE]

        # Quick exits
        if pd.isna(Y) or pd.isna(L) or L <= 0:
            adjusted_vacc = 0.0
            lo, hi = parse_year_range(row[COL_RANGE])
            if lo is None:
                lo = hi = int(Y) if pd.notna(Y) else None
            new_range = format_year_range(lo, hi) if lo is not None else ""
            denom = (row[COL_POP] or 0) + (row[COL_SLAUG] or 0)
            denom = np.nan if (pd.isna(row[COL_POP]) and pd.isna(row[COL_SLAUG])) else denom
            cov = adjusted_vacc / denom if (denom and denom != 0 and not pd.isna(denom)) else np.nan
            return pd.Series({COL_VACC: adjusted_vacc, COL_RANGE: new_range, "Vaccine Coverage": cov})

        key = (iso, dis)
        gi = group_info.get(key, {"years_asc": [], "vacc_by_year": {}, "range_by_year": {}})
        years_asc   = [y for y in gi["years_asc"] if y <= int(Y)]
        vacc_by_year = gi["vacc_by_year"]
        range_by_year= gi["range_by_year"]

        # If the current year isn't available (and should exist), fallback to current record value for Y
        # by injecting it into the maps so allocation has at least one anchor.
        if int(Y) not in years_asc:
            years_asc = sorted(years_asc + [int(Y)])
            vacc_by_year[int(Y)]  = float(0.0 if pd.isna(row[COL_VACC]) else row[COL_VACC])
            range_by_year[int(Y)] = str(row[COL_RANGE])

        # Build target window weights from lifespan
        L_float = float(L)
        n_full  = int(np.floor(L_float))
        frac    = L_float - n_full
        targets = [(int(Y) - i, 1.0) for i in range(n_full)]
        if frac > 0:
            targets.append((int(Y) - n_full, frac))

        #Reassign weights to available years using neighbor-averaging for missing years
        weights_by_year = defaultdict(float)
        for t, w in targets:
            if t in years_asc:
                weights_by_year[t] += w
                continue
            # Missing: split to nearest available on each side
            p = nearest_past(years_asc, t)
            f = nearest_future(years_asc, t, cap=int(Y))
            if p is not None and f is not None:
                weights_by_year[p] += w / 2.0
                weights_by_year[f] += w / 2.0
            elif p is not None:
                weights_by_year[p] += w
            elif f is not None:
                weights_by_year[f] += w
            else:
                # Extreme fallback: give to current year
                weights_by_year[int(Y)] += w

        # Compute weighted vaccinated sum
        adjusted_vacc = 0.0
        used_years = []
        for y, w in weights_by_year.items():
            if w == 0:
                continue
            v = vacc_by_year.get(y, 0.0)
            v = 0.0 if pd.isna(v) else float(v)
            adjusted_vacc += v * float(w)
            used_years.append(y)

        # Build combined Year Range from USED years' ranges; collapse to [single] if only one year was used (rare)
        lows, highs = [], []
        for y in used_years:
            lo, hi = parse_year_range(range_by_year.get(y))
            if lo is not None: lows.append(lo)
            if hi is not None: highs.append(hi)

        if lows and highs:
            new_range = format_year_range(min(lows), max(highs))
        else:
            # Fallback: use current row's own range or current year
            lo0, hi0 = parse_year_range(row[COL_RANGE])
            if lo0 is None:
                lo0 = hi0 = int(Y)
            new_range = format_year_range(lo0, hi0)

        # Coverage using current-row denominator
        denom = row[COL_POP] + row[COL_SLAUG]
        cov = adjusted_vacc / denom

        return pd.Series({COL_VACC: adjusted_vacc, COL_RANGE: new_range, "Vaccine Coverage": cov})

    #Apply this row-wise
    res = df.apply(adjust_for_row, axis=1)

    out = df.copy()
    out[COL_VACC] = res[COL_VACC]
    out[COL_RANGE] = res[COL_RANGE]
    out["Vaccine Coverage"] = res["Vaccine Coverage"]
    return out

In [22]:
final_dfs=[]

for year in range(2005,2026): 
    
    df2=df_result.copy()
    df2=df2[df2['Year']<=year]
    
    df_sorted = df2.sort_values(['Country', 'Disease', 'Year'], ascending=[True, True, False])
    
    cols_keep=['ISO3','Country','Year','Region','Disease','Animal category','Species','Vaccine type','Number of vaccinated','time',
               'Year Range','TOTAL Population','TOTAL Slaughtered Population','Derived_Vaccinated_Method','cattle_lifespan']
        
    # Below we account for handul of diseases that require only once-per-lifetime vaccination, not annual
    df_coverage_lifetime = df_sorted[df_sorted['Disease'].isin(once_lifetime_vaccines)]
    df_coverage_lifetime = df_coverage_lifetime.loc[:, cols_keep]
    
    max_years = df_coverage_lifetime.groupby(['ISO3', 'Disease'])['Year'].transform('max')

    # Replace "Year" with the variable 'year' only if it's the max year for that group
    # This way, we force use of most recent data reported up to that year. We look back however many years data is available for as well, from this 'year' starting point
    df_coverage_lifetime.loc[df_coverage_lifetime['Year'] == max_years, 'Year'] = year
    df_coverage_lifetime=vaccination_coverage_lifetime_vaccine(df_coverage_lifetime)
    #Only keep data relevant to the current year of interest
    df_coverage_lifetime=df_coverage_lifetime[df_coverage_lifetime['Year']==year]
    
    # Identify the latest available year for each 'Country' and 'Disease'
    latest_years = df_sorted.groupby(['Country', 'Disease'], as_index=False)['Year'].max()
    
    # Filter rows corresponding to the latest year for each 'Country' and 'Disease'
    df_latest = df_sorted.merge(latest_years, on=['Country', 'Disease', 'Year'])
    
    df_coverage_annual = df_latest[~df_latest['Disease'].isin(once_lifetime_vaccines)]
    df_coverage_annual=df_coverage_annual.loc[:,cols_keep]
    df_coverage_annual.reset_index(drop=True, inplace=True)
    
    df_coverage_annual['Year']=[year]*len(df_coverage_annual)
    
    df_coverage_annual['Vaccine Coverage']=df_coverage_annual['Number of vaccinated']/(df_coverage_annual['TOTAL Population']+df_coverage_annual['TOTAL Slaughtered Population'])
    
    df_coverage=pd.concat([df_coverage_lifetime,df_coverage_annual])
    
    df_coverage['Vaccine Coverage']=[i if i<=1 else 1 if i==i else np.nan for i in df_coverage['Vaccine Coverage']]
    df_coverage['Source'] = df_coverage['Derived_Vaccinated_Method'].apply(
        lambda x: 'WAHIS country-level report; FAOSTAT (linear interpolation between years)' if x != 'None' else 'WAHIS country-level report; FAOSTAT'
    )
    df_coverage.drop(columns=['Derived_Vaccinated_Method'],inplace=True)

    
    df_coverage=df_coverage[df_coverage['Vaccine Coverage']>=0] 
    
    VC_lower=[]
    VC_upper=[]
    
    for row in df_coverage.iterrows():
            l,u=binomtest(int(round(row[1]['Vaccine Coverage']*(row[1]['TOTAL Population']+row[1]['TOTAL Slaughtered Population']))),int(round((row[1]['TOTAL Population']+row[1]['TOTAL Slaughtered Population'])))).proportion_ci()
            VC_lower+=[l]
            VC_upper+=[u]
    
    df_coverage['Vaccine Coverage Lower']=VC_lower
    df_coverage['Vaccine Coverage Upper']=VC_upper
    
    df_part=df_part_full.copy()
    
    df_part=df_part[df_part['Year']<=year]
    
    df_part=df_part.drop_duplicates(subset=['Disease', 'Country'], keep='last')
    
    
    codes_part = [countries.get(country, 'Unknown code:'+country) for country in df_part['Country']]
    
    for code in codes_part:
        if "Unknown" in code:
            print("FIX THIS:",code)
        
    iso3s_part=[]
    
    for i in df_part['Country']:
        try:
            iso3s_part+=[countries[i]]
        except:
            iso3s_part+=[None]
    
    df_part['ISO3']=iso3s_part
    
    #print(np.unique(codes_part))
    
    index_keep=[]
    
    for row in df_part.iterrows():
        if (row[1]['Disease'] not in df_coverage[df_coverage['ISO3']==row[1]['ISO3']]['Disease'].values):
            index_keep+=[row[0]]
    df_part=df_part.loc[index_keep] 
    df_part['Vaccine Coverage']=[i if i<=1 else 1 if i==i else np.nan for i in df_part['Vaccine Coverage']]
    df_part=df_part[df_part['Vaccine Coverage']>=0]
    
    VC_lower=[]
    VC_upper=[]
    
    for row in df_part.iterrows():
            l,u=binomtest(int(round(row[1]['Vaccine Coverage']*(row[1]['Adjusted_Susceptible']))),int(round(row[1]['Adjusted_Susceptible']))).proportion_ci()
            VC_lower+=[l]
            VC_upper+=[u]
    
    df_part['Vaccine Coverage Lower']=VC_lower
    df_part['Vaccine Coverage Upper']=VC_upper
    
    add_to_coverage_df=[]
    
    for row in df_part.iterrows():
       
        if row[1]['Derived_Vaccinated_Method']!='None':
            add_to_coverage_df+=[[row[1]['ISO3'],row[1]['Country'],year,None,
                             row[1]['Disease'], None, None,None,
                     None, None,  row[1]['Year Range'], None, None,None,
                             row[1]['Vaccine Coverage'],
                     'WAHIS administrative division reports (includes linear interpolation)',row[1]['Vaccine Coverage Lower'],
                     row[1]['Vaccine Coverage Upper']]]    

        else:
                  
            add_to_coverage_df+=[[row[1]['ISO3'],row[1]['Country'],year,None,
                             row[1]['Disease'], None, None,None,
                     None, None,  row[1]['Year Range'], None, None, None,
                             row[1]['Vaccine Coverage'],
                     'WAHIS administrative division reports',row[1]['Vaccine Coverage Lower'],
                     row[1]['Vaccine Coverage Upper']]]    
       
    add_to_coverage_df = pd.DataFrame(data=add_to_coverage_df,columns=df_coverage.columns)
    
    df_coverage = pd.concat([df_coverage, add_to_coverage_df], ignore_index=True)
    

    final_dfs+=[df_coverage]

    print(year, 'finished analysis')

2005 finished analysis
2006 finished analysis
2007 finished analysis
2008 finished analysis
2009 finished analysis
2010 finished analysis
2011 finished analysis
2012 finished analysis
2013 finished analysis
2014 finished analysis
2015 finished analysis
2016 finished analysis
2017 finished analysis
2018 finished analysis
2019 finished analysis
2020 finished analysis
2021 finished analysis
2022 finished analysis
2023 finished analysis
2024 finished analysis
2025 finished analysis


In [23]:
years_data=pd.concat(final_dfs)


In [24]:
years_data=years_data.sort_values(by=['ISO3','Year','Disease','Vaccine Coverage'])

In [25]:
replacement_df=pd.read_csv(os.path.join(notebook_dir,'Common Source Data','Literature Data','FDA USA 2017 cattle vaccination coverage.csv'))
replacement_df = replacement_df.reindex(columns=years_data.columns)


# Replace rows in the original DataFrame
for _, new_row in replacement_df.iterrows():
    condition = (
        (years_data['Year'] >= new_row['Year']) & 
        (years_data['ISO3'] == new_row['ISO3']) & 
        (years_data['Disease'] == new_row['Disease'])
    )

    if years_data.loc[condition].shape[0] > 0:
        for col in years_data.columns:
            if col=='Year':
                years_data.loc[condition, col]=years_data.loc[condition, col]
            else:
                years_data.loc[condition, col] = new_row[col]
        print(f'Found and replaced values for: {new_row["Country"]} ({new_row["Disease"]})')

    else:
        # Append the new row if no match is found
        for year_add in  range(new_row['Year'], 2026):
            new_row_combined = new_row[0:2].tolist() + [year_add] + new_row[3:].tolist()
            years_data = pd.concat(
                [years_data, pd.DataFrame([new_row_combined], columns=years_data.columns)],
                ignore_index=True
            )
            print(f'Added new row for: {year_add} {new_row["Country"]} ({new_row["Disease"]})')



Found and replaced values for: United States of America (Brucella abortus (Inf. with))
Added new row for: 2017 United States of America (Brucella melitensis (Inf. with))
Added new row for: 2018 United States of America (Brucella melitensis (Inf. with))
Added new row for: 2019 United States of America (Brucella melitensis (Inf. with))
Added new row for: 2020 United States of America (Brucella melitensis (Inf. with))
Added new row for: 2021 United States of America (Brucella melitensis (Inf. with))
Added new row for: 2022 United States of America (Brucella melitensis (Inf. with))
Added new row for: 2023 United States of America (Brucella melitensis (Inf. with))
Added new row for: 2024 United States of America (Brucella melitensis (Inf. with))
Added new row for: 2025 United States of America (Brucella melitensis (Inf. with))
Added new row for: 2017 United States of America (Bovine viral diarrhoea (2006-))
Added new row for: 2018 United States of America (Bovine viral diarrhoea (2006-))
Ad

In [26]:
years_data['Disease']=[i if i!= 'Newcastle disease virus (Inf. with)' else 'Newcastle disease (velogenic)' for i in years_data['Disease']]
years_data = years_data[~((years_data['ISO3'] == 'SCG') & (years_data['Year'] > 2006))]

years_data.to_csv('2005-2025_full_cattle_vaccine_coverage_by_country.csv',index=False)
